# 05 — Forward Propagation, Logits, Softmax, and Prediction (TensorFlow / Keras)


> **Learning contract.** Every code cell is preceded by an explanation of what the code does, why the operation exists mathematically, what tensor/array shapes are expected, and what production or business failure it prevents. Run the notebooks in numerical order in a fresh Conda environment.


Forward propagation is deterministic given inputs and parameters. For our MLP: $Z_1=XW_1+b_1$, $A_1=ReLU(Z_1)$, $Z_2=A_1W_2+b_2$. $Z_2$ contains **logits**, not probabilities. Softmax converts a vector of unbounded class scores into positive values summing to one.

$softmax(z_i)=\frac{e^{z_i}}{\sum_j e^{z_j}}$. The predicted class is `argmax(logits)`; softmax does not change the argmax.


## Code walkthrough — trace four real MNIST samples through the untrained network
The code prints shapes at each stage and shows the final probability matrix. At initialization, predictions are mostly arbitrary. That is expected: architecture supplies capacity, not knowledge.


In [1]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

tf.random.set_seed(42)
from pathlib import Path
import urllib.request
import numpy as np
from sklearn.model_selection import train_test_split

SEED = 42
np.random.seed(SEED)
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = ROOT / "data"
ARTIFACT_DIR = ROOT / "artifacts"
DATA_DIR.mkdir(exist_ok=True)
ARTIFACT_DIR.mkdir(exist_ok=True)
MNIST_PATH = DATA_DIR / "mnist.npz"
MNIST_URL = "https://storage.googleapis.com/tensorflow/tf-keras-datasets/mnist.npz"


def load_official_mnist():
    if not MNIST_PATH.exists():
        print("Downloading official MNIST archive to", MNIST_PATH)
        urllib.request.urlretrieve(MNIST_URL, MNIST_PATH)
    with np.load(MNIST_PATH) as data:
        return (data["x_train"], data["y_train"], data["x_test"], data["y_test"])


def balanced_subset(x, y, per_class, seed=SEED):
    rng = np.random.default_rng(seed)
    selected = []
    for cls in range(10):
        candidates = np.flatnonzero(y == cls)
        selected.extend(rng.choice(candidates, size=per_class, replace=False))
    selected = np.asarray(selected)
    rng.shuffle(selected)
    return (x[selected], y[selected])


def prepare_splits():
    x_train_raw, y_train_raw, x_test_raw, y_test_raw = load_official_mnist()
    x_dev, y_dev = balanced_subset(x_train_raw, y_train_raw, per_class=600)
    x_test, y_test = balanced_subset(
        x_test_raw, y_test_raw, per_class=100, seed=SEED + 1
    )
    x_train, x_val, y_train, y_val = train_test_split(
        x_dev, y_dev, test_size=1000, random_state=SEED, stratify=y_dev
    )

    def transform(x):
        return x.reshape(len(x), -1).astype("float32") / 255.0

    return (
        transform(x_train),
        y_train,
        transform(x_val),
        y_val,
        transform(x_test),
        y_test,
    )


@tf.keras.utils.register_keras_serializable()
class MNISTMLP(tf.keras.Model):

    def __init__(self, hidden=64, **kwargs):
        super().__init__(**kwargs)
        self.hidden = hidden
        self.fc1 = tf.keras.layers.Dense(
            hidden, activation=None, kernel_initializer="he_normal"
        )
        self.act = tf.keras.layers.ReLU()
        self.fc2 = tf.keras.layers.Dense(
            10, activation=None, kernel_initializer="glorot_uniform"
        )

    def call(self, x, training=False):
        z1 = self.fc1(x)
        a1 = self.act(z1)
        logits = self.fc2(a1)
        return logits


model = MNISTMLP(hidden=64)
_ = model(tf.zeros((1, 784), dtype=tf.float32))
X_train, y_train, X_val, y_val, X_test, y_test = prepare_splits()
xb = tf.convert_to_tensor(X_train[:4])
logits = model(xb, training=False)
probs = tf.nn.softmax(logits, axis=1).numpy()
logits = logits.numpy()
print("logits shape", logits.shape)
print("probability sums", probs.sum(axis=1))
print("true", y_train[:4])
print("pred", probs.argmax(axis=1))
print(np.round(probs, 3))

2026-09-07 18:15:21.188872: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


logits shape (4, 10)
probability sums [1.0000001 1.        1.        1.       ]
true [0 1 9 5]
pred [5 7 5 7]
[[0.04  0.069 0.141 0.092 0.109 0.194 0.044 0.135 0.11  0.066]
 [0.112 0.08  0.064 0.096 0.094 0.111 0.063 0.179 0.057 0.144]
 [0.076 0.068 0.086 0.082 0.087 0.159 0.107 0.154 0.081 0.1  ]
 [0.106 0.089 0.08  0.089 0.073 0.12  0.108 0.131 0.105 0.099]]


## Code walkthrough — numerical stability of softmax
Exponentials can overflow when logits are large. Subtracting the maximum logit leaves all probabilities unchanged because softmax is shift-invariant, while preventing huge exponentials.


In [2]:
z = np.array([[1200.0, 1198.0, 1180.0]])
shifted = z - z.max(axis=1, keepdims=True)
p = np.exp(shifted) / np.exp(shifted).sum(axis=1, keepdims=True)
print("shifted logits", shifted)
print("stable softmax", p, "sum=", p.sum())

shifted logits [[  0.  -2. -20.]]
stable softmax [[8.80797076e-01 1.19202922e-01 1.81545808e-09]] sum= 0.9999999999999999


## Business implication
A logit is model evidence, not a calibrated probability and not a business action. Production systems typically add calibration, thresholds, abstention/manual-review policies and downstream cost rules after the model score.
